# STL-10 Training Pipeline

This notebook trains and benchmarks HBCC accuracy small, HBCC accuracy medium, and ResNet-18 on STL-10.

Dataset split used by the repo configs:
- train: 4,500 images from official STL-10 train
- val: 500 images from official STL-10 train, controlled by `data.split_seed`
- test: official STL-10 test split, evaluated once on `best.pth`


In [1]:
from pathlib import Path
import codecs
import os
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "tools").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

PYTHON = sys.executable
DATA_ROOT = ROOT / "data"
STL10_CONFIGS = [
    "configs/stl10/hbcc_accuracy_small_stl10.yaml",
    "configs/stl10/hbcc_accuracy_medium_stl10.yaml",
    "configs/stl10/resnet18_stl10.yaml",
]

RUN_ENV_CHECKS = True
RUN_DATA_CHECK = True
RUN_STL10_TRAINING = True
RUN_STL10_BENCHMARKS = True
RUN_SUMMARY = True

TRAIN_OUTPUT = "runs_stl10"
BENCHMARK_OUTPUT = "results/benchmark_stl10"
BENCHMARK_WARMUP = 10
BENCHMARK_RUNS = 30
DATA_OVERRIDES = [f"data.root={DATA_ROOT.as_posix()}"]
# Example when STL-10 is already mounted and should not be downloaded:
# DATA_OVERRIDES = ["data.root=/kaggle/input/stl10", "data.download=false"]

NO_PROGRESS = False
LIVE_OUTPUT_LINES = 30
PRINT_EVERY = 5
LIMIT_TRAIN_BATCHES = None
LIMIT_VAL_BATCHES = None
LIMIT_TEST_BATCHES = None

print("root:", ROOT)
print("python:", PYTHON)


root: d:\Research\Lightweight-Context-Cluster
python: d:\Anaconda\envs\CoC\python.exe


In [2]:
from IPython.display import clear_output

def run_py(args, max_tail=LIVE_OUTPUT_LINES):
    cmd = [PYTHON, *args]
    tail = []
    current = ""
    decoder = codecs.getincrementaldecoder("utf-8")("replace")

    def render():
        clear_output(wait=True)
        print("$", " ".join(cmd))
        lines = tail[-max_tail:]
        if current:
            lines = [*lines, current]
        print("\n".join(lines))

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
        env={**os.environ, "PYTHONUTF8": "1"},
    )
    assert process.stdout is not None
    while True:
        chunk = process.stdout.read(1)
        if not chunk:
            break
        for text in decoder.decode(chunk):
            if text == "\r":
                render()
                current = ""
            elif text == "\n":
                if current:
                    tail.append(current)
                    tail = tail[-max_tail:]
                    current = ""
                render()
            else:
                current += text
    return_code = process.wait()
    if current:
        tail.append(current)
        current = ""
    render()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {' '.join(cmd)}")

def run_train(config_path):
    args = ["tools/train.py", "--config", config_path, "--output", TRAIN_OUTPUT, "--print-every", str(PRINT_EVERY)]
    for override in DATA_OVERRIDES:
        args += ["--override", override]
    if NO_PROGRESS:
        args.append("--no-progress")
    if LIMIT_TRAIN_BATCHES is not None:
        args += ["--limit-train-batches", str(LIMIT_TRAIN_BATCHES)]
    if LIMIT_VAL_BATCHES is not None:
        args += ["--limit-val-batches", str(LIMIT_VAL_BATCHES)]
    if LIMIT_TEST_BATCHES is not None:
        args += ["--limit-test-batches", str(LIMIT_TEST_BATCHES)]
    run_py(args)

def experiment_name(config_path):
    import yaml
    with (ROOT / config_path).open("r", encoding="utf-8") as handle:
        cfg = yaml.safe_load(handle)
    return cfg.get("experiment", {}).get("name", Path(config_path).stem)


In [3]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest", "tests/test_data.py", "tests/test_models.py"], max_tail=80)


$ d:\Anaconda\envs\CoC\python.exe -m pytest tests/test_data.py tests/test_models.py



In [4]:
if RUN_DATA_CHECK:
    from lightweight_hbcc.config import apply_overrides, load_config
    from lightweight_hbcc.data import build_datasets

    cfg = apply_overrides(load_config(ROOT / STL10_CONFIGS[0]), DATA_OVERRIDES)
    train_set, val_set, test_set = build_datasets(cfg["data"], include_test=True)
    print("STL-10 split sizes")
    print("train:", len(train_set))
    print("val:", len(val_set))
    print("test:", len(test_set) if test_set is not None else None)


Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
STL-10 split sizes
train: 4500
val: 500
test: 8000


In [5]:
from lightweight_hbcc.config import load_config
from lightweight_hbcc.models import build_model

for config_path in STL10_CONFIGS:
    cfg = load_config(ROOT / config_path)
    model = build_model(cfg)
    params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{config_path}: params={params:,} trainable={trainable:,}")


configs/stl10/hbcc_accuracy_small_stl10.yaml: params=1,538,618 trainable=1,535,162
configs/stl10/hbcc_accuracy_medium_stl10.yaml: params=2,840,862 trainable=2,836,254
configs/stl10/resnet18_stl10.yaml: params=11,173,962 trainable=11,173,962


In [6]:
NO_PROGRESS = False
if RUN_STL10_TRAINING:
    for config_path in STL10_CONFIGS:
        run_train(config_path)


$ d:\Anaconda\envs\CoC\python.exe tools/train.py --config configs/stl10/hbcc_accuracy_small_stl10.yaml --output runs_stl10 --print-every 5 --override data.root=d:/Research/Lightweight-Context-Cluster/data



KeyboardInterrupt: 

In [ ]:
if RUN_STL10_BENCHMARKS:
    for config_path in STL10_CONFIGS:
        name = experiment_name(config_path)
        checkpoint = f"{TRAIN_OUTPUT}/{name}/best.pth"
        if not (ROOT / checkpoint).exists():
            print("skip missing checkpoint:", checkpoint)
            continue
        run_py([
            "tools/benchmark.py",
            "--config", config_path,
            "--checkpoint", checkpoint,
            "--output", BENCHMARK_OUTPUT,
            "--batch-sizes", "1", "16", "64", "128",
            "--warmup", str(BENCHMARK_WARMUP),
            "--runs", str(BENCHMARK_RUNS),
            "--profile",
        ], max_tail=80)


In [ ]:
if RUN_SUMMARY:
    for config_path in STL10_CONFIGS:
        name = experiment_name(config_path)
        run_dir = f"{TRAIN_OUTPUT}/{name}"
        if not (ROOT / run_dir / "metrics.jsonl").exists():
            print("skip missing metrics:", run_dir)
            continue
        run_py(["tools/summarize_training.py", run_dir], max_tail=30)


In [ ]:
if RUN_SUMMARY:
    import pandas as pd

    rows = []
    for config_path in STL10_CONFIGS:
        name = experiment_name(config_path)
        metrics_path = ROOT / TRAIN_OUTPUT / name / "metrics.jsonl"
        test_path = ROOT / TRAIN_OUTPUT / name / "test_metrics.json"
        if not metrics_path.exists():
            continue
        metrics = [json.loads(line) for line in metrics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        val_rows = [row for row in metrics if "val_acc1" in row]
        best = max(val_rows, key=lambda row: row["val_acc1"])
        row = {
            "config": name,
            "best_epoch": best.get("epoch"),
            "val_acc1": best.get("val_acc1"),
            "val_loss": best.get("val_loss"),
        }
        if test_path.exists():
            test = json.loads(test_path.read_text(encoding="utf-8"))
            row.update({"test_acc1": test.get("test_acc1"), "test_loss": test.get("test_loss")})
        rows.append(row)
    pd.DataFrame(rows).sort_values("val_acc1", ascending=False) if rows else pd.DataFrame()
